# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all record sets, fields, and columns.

In [ ]:
from pprint import pprint

# List all record sets
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"  RecordSet @id: {record_set['@id']}")
    print(f"    Name: {record_set.get('name', record_set['@id'])}")
    print("    Fields and Columns:")
    for field in record_set.get('field', []):
        if isinstance(field, dict):
            print(f"      - Field @id: {field.get('@id')}, name: {field.get('name')}, dataType: {field.get('dataType')}")
        else:
            print(f"      - Field @id: {field}")
    for column in record_set.get('column', []):
        if isinstance(column, dict):
            print(f"      - Column @id: {column.get('@id')}, name: {column.get('name')}")
        else:
            print(f"      - Column @id: {column}")

# If no record sets are listed (record sets may be detected based on the package schema), enumerate via the record_sets property
if not dataset.record_sets:
    print("No explicit record sets, they may be discovered via dataset.record_sets property or inferred.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll demonstrate extracting from all available record sets. If you see only one, use its `@id`. We'll reference via its `@id` as required.

In [ ]:
# List the record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields/columns in RecordSet {record_set_id}:")
    print(df.columns.tolist())
    display(df.head())

# For subsequent steps, pick the first record set as example, or specify one below
example_record_set_id = record_set_ids[0] if record_set_ids else None
example_df = dataframes.get(example_record_set_id) if example_record_set_id else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Let's select a numeric field (for example, `MW` for molecular weight if available), filter for proteins with MW > 10000, and normalize this column.

In [ ]:
import numpy as np

# You may need to adjust field IDs to match those found in your record set
# For demonstration, let's try common mass-spec fields
if example_df is not None and not example_df.empty:
    # Guess a numeric field
    possible_numeric_fields = [c for c in example_df.columns if 'mw' in c.lower() or 'molecular' in c.lower() or example_df[c].dtype in [float, int, np.float64, np.int64]]
    if not possible_numeric_fields:
        possible_numeric_fields = [c for c in example_df.select_dtypes(include=[np.number]).columns]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]  # The chosen field name
        print(f"Using numeric field: {numeric_field}")

        threshold = 10000
        filtered_df = example_df[example_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()

        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field, for example 'Description' or similar if present
        group_by_fields = [c for c in example_df.columns if 'desc' in c.lower() or 'category' in c.lower()]
        if group_by_fields:
            group_field = group_by_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical group field found to group by.")
    else:
        print("No numeric field found in the selected record set.")
else:
    print("No data found for the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we'll plot the distribution of a numeric field (e.g., Molecular Weight).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_df is not None and not example_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(example_df[numeric_field].dropna(), bins=50, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("Cannot display histogram - no numeric data found.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to programmatically load a FAIR mass-spectrometry dataset defined in Croissant, explored its structure via `@id` references, loaded data into pandas DataFrames, and performed basic filtering, normalization, and visualization.

**Next steps:**
- Further inspect specific proteins of interest by accession or abundance.
- Join across record sets (if available) via identifier fields.
- Extend the notebook with hypothesis testing or clustering using more `mlcroissant` and pandas functionality.